# Notebook 02 — Baseline Fine-tuning

Baseline DenseNet121 fine-tuning for multi-label chest X-ray classification.

This notebook is designed for Kaggle GPU and uses project modules from `src/`.


## 1. Environment Check

In [ ]:
import os
import sys
import platform
import torch

print('=' * 70)
print('Platform:', platform.platform())
print('Python  :', platform.python_version())
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))
else:
    print('GPU     : CPU mode')
print('=' * 70)

## 2. Clone or Update GitHub Repository

In [ ]:
REPO_URL = 'https://github.com/thnguyenvn/em-cl-xray-poc.git'
PROJECT_ROOT = '/kaggle/working/em-cl-xray-poc'

if not os.path.exists(PROJECT_ROOT):
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    %cd {PROJECT_ROOT}
    !git pull

%cd {PROJECT_ROOT}
sys.path.append(PROJECT_ROOT)

!git log --oneline -5

## 3. Import Project Modules

In [ ]:
import json
from pathlib import Path

import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, random_split

from src.datasets import (
    CheXpertDataset,
    get_train_transforms,
    get_eval_transforms,
)
from src.models import DenseNet121Classifier
from src.trainer import MultilabelTrainer
from src.metrics import compute_multilabel_metrics
from src.utils import set_seed, get_logger

print('Project modules loaded successfully.')

## 4. Experiment Configuration

In [ ]:
SEED = 42
set_seed(SEED)

OUTPUT_DIR = Path('/kaggle/working/outputs')
METRICS_DIR = OUTPUT_DIR / 'metrics'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
LOG_DIR = OUTPUT_DIR / 'logs'

for d in [METRICS_DIR, CHECKPOINT_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logger = get_logger(
    name='baseline_finetuning',
    log_dir=str(LOG_DIR),
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

CONFIG = {
    'seed': SEED,
    'image_size': 224,
    'batch_size': 16,
    'num_workers': 2,
    'epochs': 2,
    'learning_rate': 1e-4,
    'pretrained': True,
    'freeze_backbone': False,
    'val_ratio': 0.1,
    'max_train_samples': 5000,
    'max_valid_samples': 1000,
}

CONFIG

## 5. Kaggle Dataset Paths

Adjust these paths if your Kaggle input dataset names are different. Run `!find /kaggle/input -maxdepth 2` to inspect paths.

In [ ]:
!find /kaggle/input -maxdepth 2 | head -100

In [ ]:
CHEXPERT_ROOT = '/kaggle/input/chexpert-v10-small'
CHEXPERT_TRAIN_CSV = f'{CHEXPERT_ROOT}/train.csv'
CHEXPERT_VALID_CSV = f'{CHEXPERT_ROOT}/valid.csv'

print('CHEXPERT_ROOT     :', CHEXPERT_ROOT)
print('TRAIN CSV exists  :', os.path.exists(CHEXPERT_TRAIN_CSV), CHEXPERT_TRAIN_CSV)
print('VALID CSV exists  :', os.path.exists(CHEXPERT_VALID_CSV), CHEXPERT_VALID_CSV)

## 6. Define Target Labels

In [ ]:
TARGET_LABELS = [
    'No Finding',
    'Atelectasis',
    'Cardiomegaly',
    'Edema',
    'Pleural Effusion',
]

NUM_CLASSES = len(TARGET_LABELS)
print('Target labels:', TARGET_LABELS)

## 7. Load CheXpert Dataset

In [ ]:
train_transform = get_train_transforms(image_size=CONFIG['image_size'])
eval_transform = get_eval_transforms(image_size=CONFIG['image_size'])

full_train_dataset = CheXpertDataset(
    root_dir=CHEXPERT_ROOT,
    csv_file=CHEXPERT_TRAIN_CSV,
    target_labels=TARGET_LABELS,
    transform=train_transform,
    uncertainty_policy='zero',
)

print('Full train dataset:', len(full_train_dataset))

# PoC speed limit for Kaggle runs
if CONFIG['max_train_samples'] is not None:
    max_n = min(CONFIG['max_train_samples'], len(full_train_dataset))
    full_train_dataset = torch.utils.data.Subset(full_train_dataset, list(range(max_n)))

print('PoC train subset:', len(full_train_dataset))

## 8. Train/Validation Split

In [ ]:
val_size = int(len(full_train_dataset) * CONFIG['val_ratio'])
train_size = len(full_train_dataset) - val_size

generator = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=generator,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=False,
    num_workers=CONFIG['num_workers'],
    pin_memory=True,
)

print('Train size:', len(train_dataset))
print('Val size  :', len(val_dataset))

## 9. Build Model

In [ ]:
model = DenseNet121Classifier(
    num_classes=NUM_CLASSES,
    pretrained=CONFIG['pretrained'],
    freeze_backbone=CONFIG['freeze_backbone'],
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
)

trainer = MultilabelTrainer(
    model=model,
    device=DEVICE,
    optimizer=optimizer,
    logger=logger,
)

print(model.__class__.__name__)
print('Feature dim:', model.feature_dim)

## 10. Train Baseline

In [ ]:
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=CONFIG['epochs'],
    metric_fn=compute_multilabel_metrics,
)

history_df = pd.DataFrame(history)
history_df

## 11. Save Metrics and Checkpoint

In [ ]:
metrics_path = METRICS_DIR / 'baseline_finetuning_history.csv'
history_df.to_csv(metrics_path, index=False)

config_path = METRICS_DIR / 'baseline_finetuning_config.json'
with open(config_path, 'w') as f:
    json.dump(CONFIG, f, indent=2)

checkpoint_path = CHECKPOINT_DIR / 'baseline_densenet121.pth'
torch.save(
    {
        'model_state_dict': model.state_dict(),
        'config': CONFIG,
        'target_labels': TARGET_LABELS,
        'history': history,
    },
    checkpoint_path,
)

print('Saved metrics   :', metrics_path)
print('Saved config    :', config_path)
print('Saved checkpoint:', checkpoint_path)

## 12. Final Evaluation Snapshot

In [ ]:
y_true, y_prob = trainer.evaluate(val_loader)
final_metrics = compute_multilabel_metrics(y_true, y_prob)
final_metrics_df = pd.DataFrame([final_metrics])

final_metrics_path = METRICS_DIR / 'baseline_finetuning_final_metrics.csv'
final_metrics_df.to_csv(final_metrics_path, index=False)

print('Saved final metrics:', final_metrics_path)
final_metrics_df

## 13. Output Files

In [ ]:
!find /kaggle/working/outputs -maxdepth 3 -type f